In [1]:
def get_leading_trailing(grammar):
    """
    Computes LEADING and TRAILING sets for an operator grammar.
    """
    non_terminals = list(grammar.keys())
    leading = {nt: set() for nt in non_terminals}
    trailing = {nt: set() for nt in non_terminals}

    def is_non_terminal(sym):
        return sym in grammar

    # --- Compute LEADING ---
    changed = True
    while changed:
        changed = False
        for nt, productions in grammar.items():
            for prod in productions:
                # Rule 1: A -> a... or A -> Ba...
                if not is_non_terminal(prod[0]):
                    if prod[0] not in leading[nt]:
                        leading[nt].add(prod[0])
                        changed = True
                elif len(prod) > 1 and not is_non_terminal(prod[1]):
                    if prod[1] not in leading[nt]:
                        leading[nt].add(prod[1])
                        changed = True

                # Rule 2: A -> B... then LEADING(B) is in LEADING(A)
                if is_non_terminal(prod[0]):
                    prev_len = len(leading[nt])
                    leading[nt].update(leading[prod[0]])
                    if len(leading[nt]) > prev_len:
                        changed = True

    # --- Compute TRAILING ---
    changed = True
    while changed:
        changed = False
        for nt, productions in grammar.items():
            for prod in productions:
                # Rule 1: A -> ...a or A -> ...aB
                if not is_non_terminal(prod[-1]):
                    if prod[-1] not in trailing[nt]:
                        trailing[nt].add(prod[-1])
                        changed = True
                elif len(prod) > 1 and not is_non_terminal(prod[-2]):
                    if prod[-2] not in trailing[nt]:
                        trailing[nt].add(prod[-2])
                        changed = True

                # Rule 2: A -> ...B then TRAILING(B) is in TRAILING(A)
                if is_non_terminal(prod[-1]):
                    prev_len = len(trailing[nt])
                    trailing[nt].update(trailing[prod[-1]])
                    if len(trailing[nt]) > prev_len:
                        changed = True

    return leading, trailing

# --- Test Case ---
# E -> E+T | T, T -> T*F | F, F -> (E) | i
operator_grammar = {
    'E': ['E+T', 'T'],
    'T': ['T*F', 'F'],
    'F': ['(E)', 'i']
}

lead, trail = get_leading_trailing(operator_grammar)

print(f"{'NT':<5} | {'LEADING':<20} | {'TRAILING'}")
print("-" * 50)
for nt in operator_grammar:
    print(f"{nt:<5} | {str(sorted(list(lead[nt]))):<20} | {str(sorted(list(trail[nt])))}")

NT    | LEADING              | TRAILING
--------------------------------------------------
E     | ['(', '*', '+', 'i'] | [')', '*', '+', 'i']
T     | ['(', '*', 'i']      | [')', '*', 'i']
F     | ['(', 'i']           | [')', 'i']
